# D. Training Loop

- **D1**: Data shuffle every epoch
- **D2**: Standardization: Train fit -> Val transform
- **D3**: Batch size effect
- **D4**: Random seed reproducibility

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from network import Network, NetworkConfig
from optimizer import Adam

## D1: Verify Shuffle Every Epoch

In [ ]:
np.random.seed(42)
n = 20
orders = []
for epoch in range(5):
    idx = np.arange(n)
    np.random.shuffle(idx)
    orders.append(idx.copy())
    print(f'Epoch {epoch}: {idx[:10]}...')

all_diff = all(not np.array_equal(orders[i], orders[j])
               for i in range(len(orders)) for j in range(i+1, len(orders)))
print(f'\nAll orderings different: {all_diff}')
print(f'Status: {"PASS" if all_diff else "FAIL"}')

## D2: Standardization Order

In [ ]:
np.random.seed(42)
X = np.random.randn(100, 5) * 10 + 50
idx = np.arange(100); np.random.shuffle(idx)
X_train, X_val = X[idx[:80]], X[idx[80:]]

mean_tr = X_train.mean(axis=0)
std_tr = X_train.std(axis=0) + 1e-8
X_train_n = (X_train - mean_tr) / std_tr
X_val_n = (X_val - mean_tr) / std_tr  # uses TRAIN stats

print('Train normalized mean:', X_train_n.mean(axis=0).round(4))
print('Train normalized std: ', X_train_n.std(axis=0).round(4))
print('Val normalized mean:  ', X_val_n.mean(axis=0).round(4))
print('Val normalized std:   ', X_val_n.std(axis=0).round(4))

ok = np.allclose(X_train_n.mean(axis=0), 0, atol=0.1) and np.allclose(X_train_n.std(axis=0), 1, atol=0.1)
print(f'\nStatus: {"PASS" if ok else "FAIL"}')

## D3: Batch Size Effect

In [ ]:
np.random.seed(42)
x = np.random.randn(100, 30)
y = np.zeros((100, 2))
y[np.arange(100), np.random.randint(0, 2, 100)] = 1

batch_sizes = [1, 8, 32, 100]
plt.figure(figsize=(10, 5))

for bs in batch_sizes:
    np.random.seed(42)
    config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
        loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
    net = Network(config)
    opt = Adam(learning_rate=0.001)
    losses = []
    for epoch in range(100):
        idx = np.arange(len(x)); np.random.shuffle(idx)
        xs, ys = x[idx], y[idx]
        for i in range(0, len(x), bs):
            net.forward(xs[i:i+bs])
            nw, nb = net.backward(ys[i:i+bs])
            opt.update(net, nw, nb)
        out = net.forward(x)
        losses.append(net.loss(y, out))
    print(f'batch_size={bs:>4}: final loss = {losses[-1]:.6f}')
    plt.plot(losses, label=f'bs={bs}')

plt.title('Batch Size Comparison'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True); plt.show()

## D4: Seed Reproducibility

In [ ]:
def train_run(seed):
    np.random.seed(seed)
    config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
        loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
    net = Network(config)
    opt = Adam(learning_rate=0.001)
    x = np.random.randn(50, 30)
    y = np.zeros((50, 2))
    y[np.arange(50), np.random.randint(0, 2, 50)] = 1
    for _ in range(50):
        net.forward(x)
        nw, nb = net.backward(y)
        opt.update(net, nw, nb)
    return net.loss(y, net.forward(x)), net.weights

loss1, w1 = train_run(42)
loss2, w2 = train_run(42)

print(f'Run 1 loss: {loss1:.10f}')
print(f'Run 2 loss: {loss2:.10f}')
print(f'Losses match:  {loss1 == loss2}')
print(f'Weights match: {all(np.array_equal(a, b) for a, b in zip(w1, w2))}')
print(f'Status: {"PASS" if loss1 == loss2 else "FAIL"}')